## MODELOS ENTRENADOS

In [1]:
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
import numpy as np

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

# TARDA MUCHO PORQUE TIENE QUE COGER TODOS LOS PUNTOS DE LA MÁSCARA
"""def assd(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    
    d_pred_to_gt = cKDTree(gt_points).query(pred_points)[0].mean()
    d_gt_to_pred = cKDTree(pred_points).query(gt_points)[0].mean()
    
    return (d_pred_to_gt + d_gt_to_pred) / 2"""

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [10]:
import time
import torch

def measure_inference(predictor, image, point_coords, point_labels):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    
    start   = time.time()
    predictor.set_image(image)
    masks, scores, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
    latency = (time.time() - start) * 1000  # Está en ms

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return masks, scores, latency, vram

def measure_inference_sam3_prompt(predictor, img_path, text_prompt):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    start = time.time()
    predictor.set_image(img_path)
    predictor.model.set_classes(text=[text_prompt])
    predictor.prompts["text"] = [text_prompt]
    results = predictor()
    latency = (time.time() - start) * 1000
    vram = torch.cuda.memory_allocated() / 1024**2 - vram_before if torch.cuda.is_available() else 0
    return results, latency, vram

In [3]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

In [5]:
import os
import shutil
import random

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"
OUTPUT_ROOT  = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016_split"

TRAIN_IMAGES = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data")
TRAIN_MASKS  = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth")
TEST_IMAGES  = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data")
TEST_MASKS   = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth")

def split_dataset(train_ratio=0.85):
    train_names = [
        f.replace("_Segmentation.png", "")
        for f in os.listdir(TRAIN_MASKS) if f.endswith("_Segmentation.png")
    ]
    test_names = [
        f.replace("_Segmentation.png", "")
        for f in os.listdir(TEST_MASKS) if f.endswith("_Segmentation.png")
    ]

    random.seed(42)
    random.shuffle(train_names)
    n_train = int(len(train_names) * train_ratio)

    splits = {
        "train": [(n, TRAIN_IMAGES, TRAIN_MASKS) for n in train_names[:n_train]],
        "val":   [(n, TRAIN_IMAGES, TRAIN_MASKS) for n in train_names[n_train:]],
        "test":  [(n, TEST_IMAGES,  TEST_MASKS)  for n in test_names],
    }

    for split, entries in splits.items():
        os.makedirs(os.path.join(OUTPUT_ROOT, "images", split), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_ROOT, "masks",  split), exist_ok=True)
        for name, img_src_dir, mask_src_dir in entries:
            shutil.copy(
                os.path.join(img_src_dir,  name + ".jpg"),
                os.path.join(OUTPUT_ROOT, "images", split, name + ".jpg")
            )
            shutil.copy(
                os.path.join(mask_src_dir, name + "_Segmentation.png"),
                os.path.join(OUTPUT_ROOT, "masks",  split, name + ".png")
            )

split_dataset()

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from segment_anything import sam_model_registry
import cv2
import numpy as np
import os
import json

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=1024):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        for img_filename in os.listdir(self.images_dir):
            if not img_filename.lower().endswith(".jpg"):
                continue
            name      = img_filename.replace(".jpg", "")
            img_path  = os.path.join(self.images_dir, img_filename)
            mask_path = os.path.join(self.masks_dir,  name + ".png")
            if os.path.exists(mask_path):
                self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (256, 256))
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        gt_mask_full = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask_bin  = (gt_mask_full > 127).astype(np.uint8)
        contours, _  = cv2.findContours(gt_mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        x, y, w, h   = cv2.boundingRect(np.vstack(contours))

        scale_x = self.img_size / orig_w
        scale_y = self.img_size / orig_h
        cx = (x + w / 2) * scale_x
        cy = (y + h / 2) * scale_y

        point = torch.tensor([[cx, cy]]).float()
        label = torch.tensor([1])

        return image, mask, point, label


def train_sam(dataset_path, weights_path, output_weights, vit, epochs=50, batch_size=4):
    
    sam = sam_model_registry[vit](checkpoint=weights_path)
    sam.to(device)

    for param in sam.image_encoder.parameters():
        param.requires_grad = False
    for param in sam.prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam.mask_decoder.parameters(), lr=1e-4)
    scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train")
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    sam.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            images, masks  = images.to(device), masks.to(device)
            points, labels = points.to(device), labels.to(device)

            loss_batch = 0
            with torch.no_grad():
                image_embeddings = sam.image_encoder(images)

            for i in range(images.shape[0]):
                with torch.no_grad():
                    sparse_embeddings, dense_embeddings = sam.prompt_encoder(
                        points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                        boxes=None,
                        masks=None
                    )

                low_res_masks, _ = sam.mask_decoder(
                    image_embeddings=image_embeddings[i].unsqueeze(0),
                    image_pe=sam.prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save(sam.state_dict(), output_weights)
    return output_weights


## SAM 1 BASE ENTRENAMIENTO

In [ ]:
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import os
import cv2
import pandas as pd
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

DATASET_ROOT   = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016_split"
WEIGHTS        = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt"
OUTPUT_WEIGHTS = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam_b_isic2016.pt"
OUTPUT_PATH    = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"


def get_central_point(mask_bin):
    contours, _ = cv2.findContours(mask_bin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return [[x + w / 2, y + h / 2]]


def evaluate_model(dataset_path, weights_path):
    sam = sam_model_registry["vit_b"](checkpoint=weights_path)
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    iou_scores            = []
    precision_scores      = []
    recall_scores         = []
    f1_scores             = []
    dice_scores           = []
    specificity_scores    = []
    f2_scores             = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores        = []
    vram_scores           = []

    test_images_dir = os.path.join(dataset_path, "images", "test")
    test_masks_dir  = os.path.join(dataset_path, "masks",  "test")

    for img_filename in sorted(os.listdir(test_images_dir)):
        if not img_filename.lower().endswith(".jpg"):
            continue

        name      = img_filename.replace(".jpg", "")
        img_path  = os.path.join(test_images_dir, img_filename)
        mask_path = os.path.join(test_masks_dir,  name + ".png")

        if not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 127).astype(bool)
        if gt_mask.sum() == 0:
            continue

        central_point = get_central_point(gt_mask)
        if central_point is None:
            continue

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        masks, scores, latency, vram = measure_inference(predictor, image, np.array(central_point), np.array([1]))

        if masks is None or len(masks) == 0:
            continue

        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    resultados = {
        "modelo":              ["sam_b_isic2016"],
        "mean_iou":            [np.mean(iou_scores)],
        "mean_f1":             [np.mean(f1_scores)],
        "mean_recall":         [np.mean(recall_scores)],
        "mean_precision":      [np.mean(precision_scores)],
        "mean_dice":           [np.mean(dice_scores)],
        "mean_specificity":    [np.mean(specificity_scores)],
        "mean_f2":             [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":   [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":   [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":     [np.mean(latency_scores)],
        "mean_vram_mb":        [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    if os.path.exists(OUTPUT_PATH):
        df.to_csv(OUTPUT_PATH, mode="a", header=False, index=False)
    else:
        df.to_csv(OUTPUT_PATH, index=False)


trained_weights = train_sam(DATASET_ROOT, WEIGHTS, OUTPUT_WEIGHTS, vit="vit_b")
evaluate_model(DATASET_ROOT, trained_weights)

Epoch 1/50 - Loss: 0.2649
Epoch 2/50 - Loss: 0.2096
Epoch 3/50 - Loss: 0.1783
Epoch 4/50 - Loss: 0.1922
Epoch 5/50 - Loss: 0.1768
Epoch 6/50 - Loss: 0.1596
Epoch 7/50 - Loss: 0.1609
Epoch 8/50 - Loss: 0.1594
Epoch 9/50 - Loss: 0.1352
Epoch 10/50 - Loss: 0.1357
Epoch 11/50 - Loss: 0.1470
Epoch 12/50 - Loss: 0.1491
Epoch 13/50 - Loss: 0.1232
Epoch 14/50 - Loss: 0.1215
Epoch 15/50 - Loss: 0.1152
Epoch 16/50 - Loss: 0.1181
Epoch 17/50 - Loss: 0.1145
Epoch 18/50 - Loss: 0.1129
Epoch 19/50 - Loss: 0.1203
Epoch 20/50 - Loss: 0.1099
Epoch 21/50 - Loss: 0.0985
Epoch 22/50 - Loss: 0.0934
Epoch 23/50 - Loss: 0.0874
Epoch 24/50 - Loss: 0.0859
Epoch 25/50 - Loss: 0.0847
Epoch 26/50 - Loss: 0.0831
Epoch 27/50 - Loss: 0.0826
Epoch 28/50 - Loss: 0.0812
Epoch 29/50 - Loss: 0.0789
Epoch 30/50 - Loss: 0.0751
Epoch 31/50 - Loss: 0.0734
Epoch 32/50 - Loss: 0.0707
Epoch 33/50 - Loss: 0.0710
Epoch 34/50 - Loss: 0.0693
Epoch 35/50 - Loss: 0.0677
Epoch 36/50 - Loss: 0.0674
Epoch 37/50 - Loss: 0.0685
Epoch 38/5

## SAM 1 GRANDE

In [ ]:
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import os
import cv2
import pandas as pd
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

DATASET_ROOT   = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016_split"
WEIGHTS        = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt"
OUTPUT_WEIGHTS = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam_l_isic2016.pt"
OUTPUT_PATH    = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"


def get_central_point(mask_bin):
    contours, _ = cv2.findContours(mask_bin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return [[x + w / 2, y + h / 2]]


def evaluate_model(dataset_path, weights_path):
    sam = sam_model_registry["vit_l"](checkpoint=weights_path)
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    iou_scores            = []
    precision_scores      = []
    recall_scores         = []
    f1_scores             = []
    dice_scores           = []
    specificity_scores    = []
    f2_scores             = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores        = []
    vram_scores           = []

    test_images_dir = os.path.join(dataset_path, "images", "test")
    test_masks_dir  = os.path.join(dataset_path, "masks",  "test")

    for img_filename in sorted(os.listdir(test_images_dir)):
        if not img_filename.lower().endswith(".jpg"):
            continue

        name      = img_filename.replace(".jpg", "")
        img_path  = os.path.join(test_images_dir, img_filename)
        mask_path = os.path.join(test_masks_dir,  name + ".png")

        if not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 127).astype(bool)
        if gt_mask.sum() == 0:
            continue

        central_point = get_central_point(gt_mask)
        if central_point is None:
            continue

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        masks, scores, latency, vram = measure_inference(predictor, image, np.array(central_point), np.array([1]))

        if masks is None or len(masks) == 0:
            continue

        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    resultados = {
        "modelo":              ["sam_l_isic2016"],
        "mean_iou":            [np.mean(iou_scores)],
        "mean_f1":             [np.mean(f1_scores)],
        "mean_recall":         [np.mean(recall_scores)],
        "mean_precision":      [np.mean(precision_scores)],
        "mean_dice":           [np.mean(dice_scores)],
        "mean_specificity":    [np.mean(specificity_scores)],
        "mean_f2":             [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":   [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":   [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":     [np.mean(latency_scores)],
        "mean_vram_mb":        [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    if os.path.exists(OUTPUT_PATH):
        df.to_csv(OUTPUT_PATH, mode="a", header=False, index=False)
    else:
        df.to_csv(OUTPUT_PATH, index=False)


trained_weights = train_sam(DATASET_ROOT, WEIGHTS, OUTPUT_WEIGHTS, vit="vit_l")
evaluate_model(DATASET_ROOT, trained_weights)

Epoch 1/50 - Loss: 0.1685
Epoch 2/50 - Loss: 0.1080
Epoch 3/50 - Loss: 0.0909
Epoch 4/50 - Loss: 0.0796
Epoch 5/50 - Loss: 0.0718
Epoch 6/50 - Loss: 0.0636
Epoch 7/50 - Loss: 0.0520
Epoch 8/50 - Loss: 0.0533
Epoch 9/50 - Loss: 0.0484
Epoch 10/50 - Loss: 0.0450
Epoch 11/50 - Loss: 0.0441
Epoch 12/50 - Loss: 0.0394
Epoch 13/50 - Loss: 0.0378
Epoch 14/50 - Loss: 0.0379
Epoch 15/50 - Loss: 0.0435
Epoch 16/50 - Loss: 0.0346
Epoch 17/50 - Loss: 0.0352
Epoch 18/50 - Loss: 0.0312
Epoch 19/50 - Loss: 0.0313
Epoch 20/50 - Loss: 0.0294
Epoch 21/50 - Loss: 0.0264
Epoch 22/50 - Loss: 0.0234
Epoch 23/50 - Loss: 0.0221
Epoch 24/50 - Loss: 0.0214
Epoch 25/50 - Loss: 0.0213
Epoch 26/50 - Loss: 0.0210
Epoch 27/50 - Loss: 0.0225
Epoch 28/50 - Loss: 0.0241
Epoch 29/50 - Loss: 0.0217
Epoch 30/50 - Loss: 0.0206
Epoch 31/50 - Loss: 0.0206
Epoch 32/50 - Loss: 0.0207
Epoch 33/50 - Loss: 0.0216
Epoch 34/50 - Loss: 0.0243
Epoch 35/50 - Loss: 0.0212
Epoch 36/50 - Loss: 0.0201
Epoch 37/50 - Loss: 0.0190
Epoch 38/5